# CASMI26 metric, explained: MRR@25, InChIKey first block, tautomers and stereo

A short, practical guide to **how this competition scores a submission**, what changed with the **Oct 8 metric update**, and a few traps that cost points or crash a submission. Everything here is checked against the official metric source ([CASMI: Mean Reciprocal Rank](https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank)) and run on the real `train.parquet`.

**TL;DR**
- Each molecule scores **1 / (position of the first correct guess)**, or 0 if none of your up to **25** guesses is right. The final score is the mean over molecules.
- "Correct" means the same **metric identity**: stereo removed → RDKit tautomer canonicalization → first 14 characters of the InChIKey.
- **More than 25 guesses for a molecule makes the whole submission fail** (it is an error, not a truncation). A molecule missing from your file just scores 0.
- The provided `inchikey14` column is **not** the metric identity for about **1.6%** of training structures (tautomers). Don't use it to score yourself.
- The **Oct 8 update** strips stereo *before* tautomer canonicalization. It matters only for stereo-specified SMILES, and for those it fixes real mismatches (measured below).
- Rank 1 alone is worth more than ranks 2 and 3 together; most of a good score lives at rank 1.
- At the end: a **submission checker** and a **safe de-duplication** that frees wasted slots without ever moving a correct guess down.

## 1. Setup
The metric pins **RDKit 2026.03.3** because RDKit's tautomer canonicalizer changes between releases. We install the same wheel the metric uses (dataset `metric/rdkit-2026-3-3-wheel`) so every number below matches the scorer.

In [ ]:
import glob, os, subprocess, sys, time, importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def find_dir(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True), key=len)
    return os.path.dirname(hits[0]) if hits else None

WHEEL_DIR = find_dir('rdkit-2026.3.3-*.whl')
try:
    import rdkit
except ImportError:
    rdkit = None
if (rdkit is None or rdkit.__version__ != '2026.03.3') and WHEEL_DIR:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--no-index',
                        f'--find-links={WHEEL_DIR}', 'rdkit==2026.3.3'], capture_output=True, text=True)
    print('pip:', r.returncode, r.stderr[-300:])
    importlib.invalidate_caches()
    import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
RDLogger.DisableLog('rdApp.*')
print('RDKit', rdkit.__version__, '(metric pins 2026.03.3)')

DATA_DIR = find_dir('train.parquet') or 'data'
print('data:', DATA_DIR)

## 2. The metric in five lines
Our own re-implementation of the official scorer's logic (same order of steps), so you can score locally. Sources: the official metric notebook linked above, and the host's notes in the discussion ("Metric Update (RemoveStereochemistry)").

In [ ]:
TAUT = rdMolStandardize.TautomerEnumerator()

def metric_id(smiles, strip_stereo=True):
    # The competition's identity: stereo removed -> tautomer-canonical -> InChIKey first block (14 chars).
    # strip_stereo=False reproduces the rule used before the Oct 8 update.
    try:
        m = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
        if m is None:
            return None
        if strip_stereo:
            Chem.RemoveStereochemistry(m)
        return Chem.MolToInchiKey(TAUT.Canonicalize(m))[:14] or None
    except (ValueError, RuntimeError, TypeError):
        return None            # an unparseable guess is simply never a match

def parse_guesses(cell, max_rank=25):
    g = [s.strip() for s in str(cell).split(';')]
    g = [s for s in g if s]    # empty entries are dropped: 'A;;B' puts B at rank 2
    if not g:
        raise ValueError('no guesses')                       # the official scorer errors here
    if len(g) > max_rank:
        raise ValueError(f'{len(g)} guesses > {max_rank}')   # ... and here: the whole submission fails
    return g

def reciprocal_rank(answer_smiles, guesses_cell):
    target = metric_id(answer_smiles)
    for rank, s in enumerate(parse_guesses(guesses_cell), 1):
        if metric_id(s) == target:
            return 1.0 / rank                                # only the FIRST hit counts
    return 0.0

def mrr(solution, submission):
    # solution / submission: DataFrames with molecule_id, smiles. Missing molecules score 0.
    sub = dict(zip(submission.molecule_id, submission.smiles))
    return float(np.mean([reciprocal_rank(a, sub[m]) if m in sub else 0.0
                          for m, a in zip(solution.molecule_id, solution.smiles)]))

# the official notebook's own examples, reproduced
sol = pd.DataFrame({'molecule_id': ['m_1', 'm_2'], 'smiles': ['CCO', 'c1ccccc1']})
print(mrr(sol, pd.DataFrame({'molecule_id': ['m_1', 'm_2'], 'smiles': ['CCO;CC(=O)O', 'CC(=O)O;c1ccccc1']})))   # 0.75
print(mrr(sol, pd.DataFrame({'molecule_id': ['m_1', 'm_2'], 'smiles': ['CC(=O)O', 'c1ccccc1;c1ccccc1']})))     # 0.5
print(reciprocal_rank('C=C(C)C1CCC2CCC(=O)C=C2C1', 'C=C(C)[C@@H]1CCC2CCC(=O)C=C2C1'))                        # 1.0

## 3. What counts as "the same molecule"
Three things are ignored: **stereochemistry**, **tautomeric form** and (through InChI) most **protonation states**. A few pairs, with the plain InChIKey first block next to the metric identity:

In [ ]:
pairs = [
    ('2-pyridone vs 2-hydroxypyridine (tautomers)', 'O=c1cccc[nH]1', 'Oc1ccccn1'),
    ('acetylacetone keto vs enol (tautomers)', 'CC(=O)CC(C)=O', 'CC(=O)/C=C(/C)O'),
    ('L- vs D-alanine (stereo)', 'C[C@H](N)C(=O)O', 'C[C@@H](N)C(=O)O'),
    ('glycine neutral vs zwitterion (protomer)', 'NCC(=O)O', '[NH3+]CC(=O)[O-]'),
    ('enone pinned by a stereocentre (the Oct 8 case)', 'C=C(C)C1CCC2CCC(=O)C=C2C1', 'C=C(C)[C@@H]1CCC2CCC(=O)C=C2C1'),
]
rows = []
for name, a, b in pairs:
    ma, mb = Chem.MolFromSmiles(a), Chem.MolFromSmiles(b)
    rows.append(dict(case=name,
                     plain_ik14_same=Chem.MolToInchiKey(ma)[:14] == Chem.MolToInchiKey(mb)[:14],
                     old_rule_same=metric_id(a, False) == metric_id(b, False),
                     metric_same=metric_id(a) == metric_id(b)))
pd.DataFrame(rows)

- **Plain InChIKey** already ignores stereo (it lives in the second block) and handles some protomers, but not every tautomer pair.
- The **tautomer canonicalization** closes the tautomer gap.
- The **last row** is why the Oct 8 update exists: the stereocentre stops RDKit from moving the C=C bond, so without stripping stereo first the two land on different tautomers and different keys. Before the update that guess scored 0; now it scores 1.

## 4. How often does this matter? Measured on `train.parquet`
We take 20,000 random unique training structures and compare the **provided `inchikey14` column** with the **metric identity**.

In [ ]:
t = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'), columns=['normalized_smiles', 'inchikey14'])
u = t.drop_duplicates('normalized_smiles').reset_index(drop=True)
print(f'{len(t):,} spectra, {len(u):,} unique structures')
has_stereo = u.normalized_smiles.str.contains('[@/\\\\]', regex=True).mean()
print(f'training SMILES carrying stereo marks (@ / backslash): {has_stereo:.2%}')
s = u.sample(20000, random_state=0).reset_index(drop=True)
t0 = time.time()
s['metric_id'] = [metric_id(x) for x in s.normalized_smiles]
print(f'{time.time() - t0:.0f}s')
diff = s[s.metric_id != s.inchikey14]
print(f'metric identity != provided inchikey14: {len(diff)} of {len(s)} ({len(diff)/len(s):.2%})')
diff.head(8)

- The provided `inchikey14` is the plain InChIKey block. For about **1.6%** of structures the metric's tautomer step gives a **different** key.
- If you score a local validation by comparing `inchikey14` strings, those molecules can read as misses when the real scorer counts them as hits (or the reverse). Always score with the metric identity.
- Training SMILES carry **no stereo**, so the answers you are matched against are stereo-free.

## 5. The Oct 8 update, measured
Candidate structures from databases such as PubChem or COCONUT often **do** carry stereo. To measure the effect without any external file, we take training structures with stereocentres, attach one random stereo assignment to each (as a database entry would have), and compare the **old rule** (no stereo stripping) with the **current rule**, both against the stereo-free answer.

In [ ]:
rng = np.random.default_rng(0)
opts = StereoEnumerationOptions(onlyUnassigned=True, maxIsomers=8, rand=0, unique=True)
pool = u.sample(40000, random_state=1).normalized_smiles.tolist()
rows, t0 = [], time.time()
for smi in pool:
    m = Chem.MolFromSmiles(smi)
    if m is None:
        continue
    iso = list(EnumerateStereoisomers(m, options=opts))
    if len(iso) < 2:
        continue                                   # no stereo element to assign
    stereo_smi = Chem.MolToSmiles(iso[int(rng.integers(len(iso)))])
    ans = metric_id(smi)
    rows.append((smi, stereo_smi, metric_id(stereo_smi, strip_stereo=False) == ans, metric_id(stereo_smi) == ans))
    if len(rows) >= 6000:
        break
st = pd.DataFrame(rows, columns=['answer', 'stereo_guess', 'match_old_rule', 'match_new_rule'])
print(f'{time.time() - t0:.0f}s, {len(st)} stereo-capable structures')
print(f"old rule (before Oct 8): {1 - st.match_old_rule.mean():.2%} of correct stereo guesses scored as WRONG")
print(f"new rule:                {1 - st.match_new_rule.mean():.2%}")
st[~st.match_old_rule].head(6)

- Under the old rule a small but real share of **correct** stereo-specified guesses did not match their stereo-free answer. They are exactly the "stereocentre pins a double bond" cases.
- The new rule matches all of them. Candidate lists built from COCONUT or PubChem often carry stereo marks (in one candidate panel we checked, about one structure in five), so the fix touches many molecules a little.
- **Practical rule:** key, de-duplicate and fuse your candidates with the same identity the metric uses (stereo stripped first). Otherwise two entries the metric treats as one molecule can both take slots in your top 25.

## 6. What each rank is worth

In [ ]:
r = np.arange(1, 26)
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(r, 1 / r, color='#4C72B0')
ax.set_xlabel('rank of the first correct guess'); ax.set_ylabel('score for that molecule'); ax.set_xticks(r)
ax.set_title('MRR@25: rank 1 = 1.0, rank 2 = 0.5, rank 3 = 0.33 ... rank 25 = 0.04')
plt.tight_layout(); plt.show()
print('ranks 2 + 3 together are worth', round(1/2 + 1/3, 3), '- less than one rank-1 hit (1.0)')
print('moving one answer from rank 2 to rank 1 = +0.5 for that molecule = +0.5/N on the mean over N molecules')

- Public probes in this competition (lszlst's "top-1 only" and "top-2 only" submissions) suggest that **most of a good score comes from rank 1**. Ordering close isomers (same formula, different structure) is where the points are.
- With N scored molecules, one molecule moves the mean by at most 1/N (0.0025 for N = 400). Small leaderboard steps are a handful of molecules, so treat differences of a few thousandths with care.

## 7. A submission checker and a safe de-duplication
Run this on your `submission.csv` before you submit. It catches the hard errors the scorer raises, and the soft problems that silently cost points.

The de-duplication keeps the **first** occurrence of each metric identity and drops unparseable entries. A correct guess can only move **up** (or stay), never down, because only entries above it can disappear. The idea of a monotone, metric-identity clean-up was popularised by [OBSTACLE DEVELOPER's V1.1 notebook](https://www.kaggle.com/code/obstacledeveloper/casmi26-v1-1-metric-audit-and-safe-tail). This is our own small version.

In [ ]:
def check_submission(sub, sample=None, max_rank=25):
    issues = []
    if list(sub.columns[:2]) != ['molecule_id', 'smiles']:
        issues.append(('ERROR', f'columns should be molecule_id, smiles; got {list(sub.columns)}'))
    if sub[['molecule_id', 'smiles']].isna().any().any():
        issues.append(('ERROR', 'NaN in molecule_id or smiles'))
    if sub.molecule_id.duplicated().any():
        issues.append(('ERROR', f'{sub.molecule_id.duplicated().sum()} duplicated molecule_id'))
    if sample is not None:
        miss = set(sample.molecule_id) - set(sub.molecule_id)
        if miss:
            issues.append(('WARN', f'{len(miss)} molecules missing (they score 0)'))
    n_dup = n_bad = 0
    for mid, cell in zip(sub.molecule_id, sub.smiles.astype(str)):
        g = [x.strip() for x in cell.split(';') if x.strip()]
        if not g:
            issues.append(('ERROR', f'{mid}: no guesses')); continue
        if len(g) > max_rank:
            issues.append(('ERROR', f'{mid}: {len(g)} guesses > {max_rank} (submission fails)'))
        ids = [metric_id(x) for x in g]
        n_bad += sum(i is None for i in ids)
        n_dup += len([i for i in ids if i]) - len(set(i for i in ids if i))
    if n_bad:
        issues.append(('WARN', f'{n_bad} unparseable guesses (wasted slots)'))
    if n_dup:
        issues.append(('WARN', f'{n_dup} guesses repeat an identity already listed above them (wasted slots)'))
    return issues or [('OK', 'no problems found')]

def safe_dedup(cell, max_rank=25):
    out, seen = [], set()
    for x in (s.strip() for s in str(cell).split(';')):
        i = metric_id(x) if x else None
        if i is None or i in seen:
            continue
        seen.add(i); out.append(x)
    return ';'.join(out[:max_rank])

toy = pd.DataFrame({'molecule_id': ['a', 'b', 'c'],
                    'smiles': ['O=c1cccc[nH]1;Oc1ccccn1;CCO',          # tautomer duplicate
                               'C[C@H](N)C(=O)O;C[C@@H](N)C(=O)O;;CC',  # stereo duplicate + empty entry
                               'not_a_smiles;CCO']})                    # unparseable entry
for level, msg in check_submission(toy):
    print(f'{level:5s} {msg}')
toy['smiles_clean'] = toy.smiles.map(safe_dedup)
toy

- After `safe_dedup`, fill the freed slots with your next-best *different* candidates. An empty slot can never score.
- For your real file: `check_submission(pd.read_csv('submission.csv'), pd.read_csv(<sample_submission.csv>))`.

## Sources and credits
- Official metric: [CASMI: Mean Reciprocal Rank](https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank) (Kaggle), and the host's Oct 8 note "Metric Update (RemoveStereochemistry)".
- RDKit 2026.03.3 (`metric/rdkit-2026-3-3-wheel`): MolStandardize TautomerEnumerator, InChI.
- Monotone metric-identity clean-up idea: [OBSTACLE DEVELOPER, CASMI26 V1.1](https://www.kaggle.com/code/obstacledeveloper/casmi26-v1-1-metric-audit-and-safe-tail).
- "Most of the score is rank 1": public probe submissions discussed by lszlst in the competition forum.
- All numbers above are computed in this notebook from the competition's `train.parquet`.